## Preprocesamiento para Data Mining (Antes era Sección 4)
Ahora que los datos están limpios, pasamos a la ingeniería de características y preparación para modelos.

### 5.1 Escalado y Normalización (Actualizado con más opciones)

In [1]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:

# Datos de ejemplo
df_esc = pd.DataFrame({'Edad': [23, 35, 29, 41, 29], 'Salario': [50000, 75000, 60000, 120000, 45000]})

# StandardScaler (media 0, std 1) - PARA MODELOS LINEALES Y DISTANCIAS
scaler = StandardScaler()
df_std = pd.DataFrame(scaler.fit_transform(df_esc), columns=df_esc.columns)
print("StandardScaler:\n", df_std)

# MinMaxScaler (0 a 1) - PARA REDES NEURONALES Y ALGORITMOS CON ACTIVACIONES
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df_esc), columns=df_esc.columns)
print("\nMinMaxScaler:\n", df_minmax)

# RobustScaler (usa mediana y IQR) - CUANDO HAY OUTLIERS
robust = RobustScaler()
df_robust = pd.DataFrame(robust.fit_transform(df_esc), columns=df_esc.columns)
print("\nRobustScaler:\n", df_robust)

# PowerTransformer (Box-Cox o Yeo-Johnson) - PARA HACER DATOS MÁS NORMALES
pt = PowerTransformer(method='yeo-johnson')  # maneja negativos
df_pt = pd.DataFrame(pt.fit_transform(df_esc), columns=df_esc.columns)
print("\nPowerTransformer (Yeo-Johnson):\n", df_pt)

StandardScaler:
        Edad   Salario
0 -1.372813 -0.740233
1  0.588348  0.185058
2 -0.392232 -0.370117
3  1.568929  1.850583
4 -0.392232 -0.925292

MinMaxScaler:
        Edad   Salario
0  0.000000  0.066667
1  0.666667  0.400000
2  0.333333  0.200000
3  1.000000  1.000000
4  0.333333  0.000000

RobustScaler:
    Edad  Salario
0  -1.0     -0.4
1   1.0      0.6
2   0.0      0.0
3   2.0      2.4
4   0.0     -0.6

PowerTransformer (Yeo-Johnson):
        Edad   Salario
0 -1.495899 -0.785857
1  0.652941  0.619731
2 -0.310495 -0.057391
3  1.463947  1.520674
4 -0.310495 -1.297158


### 5.2 Codificación de Categóricas (Actualizado)


In [3]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

# One-Hot Encoding (evita relaciones ordinales)
df_cat = pd.DataFrame({'Ciudad': ['Madrid', 'Barcelona', 'Valencia', 'Madrid']})
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(df_cat[['Ciudad']])
df_ohe = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
print("One-Hot Encoder:\n", df_ohe)

# Ordinal Encoding (cuando hay orden, ej: 'Bajo', 'Medio', 'Alto')
# Solo para casos específicos, generalmente mejor One-Hot o Target Encoding.

One-Hot Encoder:
    Ciudad_Barcelona  Ciudad_Madrid  Ciudad_Valencia
0               0.0            1.0              0.0
1               1.0            0.0              0.0
2               0.0            0.0              1.0
3               0.0            1.0              0.0


### 5.3 Pipelines Completos (Uniendo todo)


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Definir columnas
num_cols = ['Edad']
cat_cols = ['Ciudad']

# Tubería para numéricas
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Tubería para categóricas
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Preprocesador final
preprocesador = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

# Aplicar al DataFrame (suponiendo que existe)
# X_limpio = preprocesador.fit_transform(df)


# Sección 6: Buenas Prácticas y Checklist Final (Actualizada)

1. **Orden de limpieza:** 1) Duplicados, 2) Tipos de datos, 3) Fechas/Strings, 4) Nulos (primero analizar, luego imputar/eliminar), 5) Outliers (primero analizar, luego transformar/capear/eliminar).
2. **NUNCA** imputes con la media antes de tratar los outliers (la media se corrompe).
3. **Siempre** guarda una copia del original (`df_original = df.copy()`) antes de empezar a limpiar.
4. **Documenta** cada decisión de limpieza (¿por qué elegiste la mediana? ¿por qué borraste esa fila?).
5. **Valida** después de limpiar: repite el `describe()` y `info()` para asegurarte de que todo tiene sentido.
6. **Si usas pipelines**, asegúrate de que el `preprocesador` se ajuste SÓLO con `X_train` y transforme a `X_test` (evita data leakage).
